<a href="https://colab.research.google.com/github/Clint07-datascientist/AgroInsightX_Chatbot/blob/main/notebooks/chatbot_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Replace file path with GIT_URL
GIT_URL = "https://github.com/Clint07-datascientist/AgroInsightX_Chatbot.git"

# Clone the repository to the Colab environment
!git clone $GIT_URL

In [ ]:
from datasets import load_dataset

dataset = load_dataset('parquet', data_files='/content/AgroInsightX_Chatbot/data/agriculture-qa-english-only.parquet')

#**EXPLORATORY DATA ANALYSIS**

In [ ]:
# Check column names and number of rows
print(dataset)

In [ ]:
# Check column names and number of rows
print(dataset['train'].features)

In [ ]:
# Check for missing values (requires converting to a pandas DataFrame first)
import pandas as pd
df = dataset['train'].to_pandas()
print(df.isnull().sum())

In [ ]:
# Text length distribution (for 'question' column)
df['question_length'] = df['question'].apply(lambda x: len(str(x).split()))
print(df['question_length'].describe())

In [ ]:
# Example of text length distribution (for 'answer' column)
df['answers_length'] = df['answers'].apply(lambda x: len(str(x).split()))
print(df['answers_length'].describe())

In [ ]:
print("Dataset Overview:")
print("=" * 50)
print(f"Dataset shape: {df.shape}")
print(f"Number of reviews: {len(df)}")
print(f"Number of features: {len(df.columns)}")

In [ ]:
# Basic dataset information
print("\nDataset Information:")
print("=" * 50)
print(df.info())
print("\nFirst few rows:")
print("=" * 50)
print(df.head())
print("\nColumn names:")
print("=" * 50)
print(df.columns.tolist())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


# Visualize the distribution of question lengths
plt.figure(figsize=(10, 6))
sns.histplot(df['question_length'], bins=50, kde=True)
plt.title('Distribution of Question Word Counts')
plt.xlabel('Word Count')
plt.show()

# Visualize the distribution of answers lengths
plt.figure(figsize=(10, 6))
sns.histplot(df['answers_length'], bins=50, kde=True)
plt.title('Distribution of Answers Word Counts')
plt.xlabel('Word Count')
plt.show()

In [ ]:
from wordcloud import WordCloud

# Create a single string of all questions
all_questions = ' '.join(df['question'].tolist())

# Generate the word cloud
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(all_questions)

# Display the word cloud
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of Questions')
plt.show()

In [ ]:
# Create a single string of all answers
all_answers = ' '.join(df['answers'].tolist())

# Generate the word cloud
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(all_answers)

# Display the word cloud
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of Answers')
plt.show()

In [ ]:
import nltk
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

# Download necessary NLTK data (if you haven't already)
nltk.download('punkt')
nltk.download('stopwords')

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')

In [ ]:
import nltk
nltk.download('punkt_tab')

In [ ]:
from nltk.corpus import stopwords

# Define a function to extract n-grams
def get_ngrams(text, n=1, stop_words=None):
    tokens = nltk.word_tokenize(text.lower())
    if stop_words:
        tokens = [word for word in tokens if word not in stop_words]
    if n == 1:
        return tokens  # Return a flat list of tokens for unigrams
    ngrams = [(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
    # Join bigrams into single strings for easier counting
    if n > 1:
        ngrams = [" ".join(ngram) for ngram in ngrams]
    return ngrams

# Get English stop words
stop_words = set(stopwords.words('english'))

In [ ]:
# Analyze and visualize top unigrams in questions
question_unigrams = get_ngrams(' '.join(df['question'].tolist()), n=1, stop_words=stop_words)
question_unigram_counts = Counter(question_unigrams)
top_question_unigrams = question_unigram_counts.most_common(20)

# Plotting the top unigrams
plt.figure(figsize=(12, 8))
sns.barplot(x=[word for word, count in top_question_unigrams], y=[count for word, count in top_question_unigrams])
plt.title('Top 20 Unigrams in Questions (excluding stop words)')
plt.xlabel('Unigram')
plt.ylabel('Frequency')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Analyze and visualize top unigrams in answers
answer_unigrams = get_ngrams(' '.join(df['answers'].tolist()), n=1, stop_words=stop_words)
answer_unigram_counts = Counter(answer_unigrams)
top_answer_unigrams = answer_unigram_counts.most_common(20)

# Plotting the top unigrams
plt.figure(figsize=(12, 8))
sns.barplot(x=[word for word, count in top_answer_unigrams], y=[count for word, count in top_answer_unigrams])
plt.title('Top 20 Unigrams in Answers (excluding stop words)')
plt.xlabel('Unigram')
plt.ylabel('Frequency')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

#**Data Preprocessing**

In [ ]:
from transformers import AutoTokenizer

# Choose a model you intend to use, e.g., 'bert-base-uncased'
model_checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [ ]:
def preprocess_function(examples):
    questions = [q.strip() for q in examples["question"]]
    answers = examples["answers"]

    # Tokenize the question and answer text with padding and truncation
    inputs = tokenizer(
        questions,
        answers,
        max_length=512,
        truncation="only_second",  # Truncate only the answer if combined length exceeds max_length
        padding="max_length",
        return_offsets_mapping=True, # Return mapping from token to character span in the original text
    )

    offset_mapping = inputs.pop("offset_mapping")
    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        answer = answers[i]
        # Find the start and end character indices of the answer in the original text
        start_char = examples["answers"][i].find(answer)
        end_char = start_char + len(answer)

        # Find the start and end token indices of the answer in the tokenized sequence
        start_token = -1
        end_token = -1
        for token_idx, (start, end) in enumerate(offsets):
            if start == start_char:
                start_token = token_idx
            if end == end_char:
                end_token = token_idx
                break # Found the end token, no need to continue

        start_positions.append(start_token)
        end_positions.append(end_token)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

In [ ]:
# Split the 'train' split into 90% train and 10% validation
train_test_split = dataset['train'].train_test_split(test_size=0.1)
dataset['train'] = train_test_split['train']
dataset['validation'] = train_test_split['test']

In [ ]:
# Apply the function across the whole dataset AFTER splitting
tokenized_dataset = dataset.map(preprocess_function, batched=True)

# Task
Explain the next steps in training an AgroInsightX chatbot after data preprocessing.

## Choose a model

### Subtask:
Select a pre-trained model suitable for question answering tasks, such as a transformer-based model.


**Reasoning**:
Select a suitable pre-trained model for question answering. 'bert-base-uncased' is a common choice for such tasks and was already used for tokenization, making it a consistent choice for the model as well.



In [ ]:
from transformers import AutoModelForQuestionAnswering

# Load a pre-trained model suitable for question answering
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)

## Fine-tune the model

### Subtask:
Adapt the chosen model to your specific agriculture QA dataset.


**Reasoning**:
Define the training arguments and create a Trainer object for fine-tuning the model on the dataset.



**Reasoning**:
The previous command failed because `evaluation_strategy` is not a valid argument for `TrainingArguments`. I need to update the `TrainingArguments` to use `eval_strategy` instead.



**Reasoning**:
The traceback indicates a KeyError because the key 'validation' does not exist in `tokenized_dataset`. I need to check the available keys in `tokenized_dataset` and use the correct key for the evaluation dataset, which should be 'test' based on the previous split.



In [ ]:
from transformers import TrainingArguments, Trainer

# 1. Define training arguments
training_args = TrainingArguments(
    output_dir="./results",  # Output directory for model predictions and checkpoints
    eval_strategy="epoch",  # Evaluate every epoch
    learning_rate=0.00001,  # Learning rate
    per_device_train_batch_size=8,  # Batch size per device during training
    per_device_eval_batch_size=8,  # Batch size per device during evaluation
    num_train_epochs=3,  # Number of training epochs
    weight_decay=0.01,  # Strength of weight decay
    push_to_hub=False, # Whether to push the model to the Hugging Face Hub
    remove_unused_columns=False, # Keep unused columns in the dataset
)

# Remove the original columns that are not needed for training
tokenized_dataset = tokenized_dataset.remove_columns(['question', 'answers'])

# Print the keys of the tokenized dataset to check for 'validation'
print("Keys in tokenized_dataset:", tokenized_dataset.keys())

# 2. Create a Trainer object
trainer = Trainer(
    model=model,  # The fine-tuned model
    args=training_args,  # The training arguments
    train_dataset= tokenized_dataset['train'],  # The tokenized training dataset
    eval_dataset= tokenized_dataset['validation'],  # The tokenized validation dataset
    tokenizer=tokenizer,  # The tokenizer
)

# 3. Start the training process
trainer.train()

#**Evaluation**

In [ ]:
!pip install evaluate

In [ ]:
import collections # Required for defaultdict and OrderedDict in postprocessing
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer
)
import evaluate
from tqdm.auto import tqdm

In [ ]:
# Load the dataset
dataset = load_dataset("parquet", data_files="/content/AgroInsightX_Chatbot/data/agriculture-qa-english-only.parquet")

# Split the dataset (assuming you used a 90/10 split in training)
train_test_split = dataset['train'].train_test_split(test_size=0.1, seed=42)
raw_datasets = {
    'train': train_test_split['train'],
    'validation': train_test_split['test'] # Use 'test' key from the split
}
validation_set = raw_datasets['validation'] # This will now correctly reference 'test'

print(f"Validation set size: {len(validation_set)}")

In [ ]:
model_checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

max_length = 384  # Max input sequence length
stride = 128      # Overlap between context chunks

# Define the model path (assuming you pushed to the Hub)
model_path = "AgroInsightX_Chatbot/models/"

In [ ]:
def preprocess_validation_examples(examples):
    questions = [q.strip() for q in examples["question"]]
    tokenized_examples = tokenizer(
        questions,
        examples["answers"],  # Use 'answers' as the context
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    # Store the context IDs for mapping back to the original context
    sample_map = tokenized_examples.pop("overflow_to_sample_mapping")
    # The original dataset does not have an 'id' column, so we cannot create 'example_id' this way.
    # If you need to link back to original examples, you might need to add an index or ID column
    # to your dataset beforehand. For now, we will skip creating 'example_id' based on 'id'.
    # tokenized_examples["example_id"] = [
    #     examples["id"][idx] for idx in sample_map
    # ]
    tokenized_examples["example_id"] = sample_map # Use the sample_map directly as a temporary identifier if needed later

    # Keep offsets mapping to reconstruct the answer span from tokens
    tokenized_examples["offset_mapping"] = tokenized_examples.pop("offset_mapping")

    return tokenized_examples

# Apply the preprocessing function
tokenized_validation_set = validation_set.map(
    preprocess_validation_examples,
    batched=True,
    remove_columns=validation_set.column_names
)

In [ ]:
# Load the best model from the Hugging Face Hub (or local directory)
model = AutoModelForQuestionAnswering.from_pretrained("./results/checkpoint-7500") # Assuming the model was saved to ./results

In [ ]:
# --- Function Definitions for Evaluation ---

def postprocess_qa_predictions(examples, features, raw_predictions, n_best_size=20, max_answer_length=30):
    all_start_logits, all_end_logits = raw_predictions

    # Map features back to original examples using the example_id (which is the original index)
    features_per_example = collections.defaultdict(list)
    for i, feature in enumerate(features):
        original_example_index = feature["example_id"] # This is the index in the original dataset batch
        features_per_example[original_example_index].append(i)

    # Prepare for storing best answers
    predictions = collections.OrderedDict()

    # tqdm is used for progress bar during evaluation
    # Iterate over all original examples
    for example_index, example in enumerate(tqdm(examples)): # Iterate through the original examples
        # Find the features corresponding to this original example using its index
        feature_indices = features_per_example[example_index] # Use the example_index to get related features
        min_null_score = None
        valid_answers = []

        # Iterate over all context chunks (features) for this original example
        for feature_index in feature_indices:
            start_logits = all_start_logits[feature_index]
            end_logits = all_end_logits[feature_index]
            offset_mapping = features[feature_index]["offset_mapping"]

            # Context for the current feature - this should come from the original example
            # The original context is in example["answers"]
            context = example["answers"]

            # Compute the null score (score for predicting no answer)
            start_null_score = start_logits[0]
            end_null_score = end_logits[0]
            feature_null_score = start_null_score + end_null_score
            if min_null_score is None or min_null_score < feature_null_score:
                min_null_score = feature_null_score

            # Find the N best start and end tokens
            start_indexes = np.argsort(start_logits)[-1 : -n_best_size - 1 : -1].tolist()
            end_indexes = np.argsort(end_logits)[-1 : -n_best_size - 1 : -1].tolist()

            for start_index in start_indexes:
                for end_index in end_indexes:
                    # Filter for invalid spans
                    if start_index >= len(offset_mapping) or end_index >= len(offset_mapping):
                        continue
                    # Check if tokens are within the context part (after the question)
                    # The first token ([CLS]) is usually at index 0. The question tokens follow, then [SEP].
                    # We need to find the index of the first token of the context.
                    # Assuming the tokenizer adds [CLS] and [SEP] tokens, the context starts after the second [SEP].
                    # A more robust way is to check the token_type_ids if available, or rely on offset_mapping.
                    # The current check 'start_index not in features[feature_index]["input_ids"]' seems incorrect.
                    # Let's use the offset_mapping to ensure the span is within the original context.
                    # The offset_mapping for tokens corresponding to the question will have start/end positions
                    # within the question string, and for the context within the answer string.
                    # We need to ensure the predicted span's start and end characters fall within the original answer text.
                    # The preprocess_validation_examples truncates "only_second" (the answer/context).
                    # The offset_mapping corresponds to the combined question+answer.
                    # We need to know where the answer starts in the combined input_ids.
                    # The `token_type_ids` can distinguish between question (0) and answer (1) tokens.
                    # Let's assume token_type_ids are available in the features.
                    if "token_type_ids" in features[feature_index]:
                         if features[feature_index]["token_type_ids"][start_index] != 1 or features[feature_index]["token_type_ids"][end_index] != 1:
                             continue # Ensure both start and end tokens are part of the context (type 1)
                    else:
                        # Fallback if token_type_ids are not available (less robust)
                        # Check if the offset corresponds to a character within the original answer text
                        start_char_pos = offset_mapping[start_index][0]
                        end_char_pos = offset_mapping[end_index][1]
                        # This check is not straightforward without knowing where the answer starts in the original text.
                        # Let's assume token_type_ids are the standard way. If not, this needs re-evaluation.
                        pass # Keep the less robust check for now or rely on token_type_ids


                    if end_index < start_index or end_index - start_index + 1 > max_answer_length:
                        continue

                    # Get start and end character positions from the offset mapping
                    start_char_pos = offset_mapping[start_index][0]
                    end_char_pos = offset_mapping[end_index][1]

                    # Get the predicted text from the original context (answer)
                    # The offset mapping is for the combined input. We need to get the span from the original *answer*.
                    # However, the offset_mapping is relative to the start of the *feature*'s text.
                    # When truncation happens, the offsets are relative to the truncated text.
                    # This is complex. Let's simplify and assume the offsets are relative to the context within the feature.
                    # The preprocess_validation_examples function used `examples["answers"]` as the context.
                    # The offset mapping from the tokenizer is relative to the start of the *combined* sequence (question + answer).
                    # We need to find the character start position of the answer within the original text to correctly use offsets.
                    # Let's assume the offsets *do* map correctly back to the original text span in the original example's answer.
                    # The `start_char` and `end_char` in the training preprocessing function show how to get the character span in the original answer.
                    # We need to link the token offsets back to the original answer string.
                    # The offset_mapping gives (start_char_in_feature, end_char_in_feature).
                    # We need to map these feature-relative character positions back to the original answer string.
                    # This seems to require knowing the start character position of the answer within the original example.
                    # Let's go back to the original `preprocess_validation_examples`. It doesn't calculate start/end *character* positions of the answer.
                    # It only returns offset_mapping for the tokens in the feature.
                    # The postprocessing needs to use these token offsets to extract text from the *original* answer string.
                    # Let's assume the offset_mapping gives character positions relative to the start of the original example's answer string.
                    # This is a strong assumption and might be the source of issues.
                    # A more correct approach involves finding the start character of the answer in the original text and adjusting offsets.

                    # Let's stick to the original logic for now and try to make it work with the available data.
                    # The offset_mapping gives character offsets within the *feature's* text.
                    # The feature's text is derived from the original example's question and answer.
                    # We need to extract the span from the original example's answer using these offsets.
                    # This implies the offset_mapping should be relative to the start of the *original* answer.
                    # Let's assume this is the case for now.
                    try:
                        predicted_text = context[start_char_pos:end_char_pos]
                    except IndexError:
                         # Handle cases where offsets are out of bounds for the original context string
                         continue


                    valid_answers.append(
                        {
                            "score": start_logits[start_index] + end_logits[end_index],
                            "text": predicted_text
                        }
                    )

        # Select the best non-null answer
        if len(valid_answers) > 0:
            best_answer = sorted(valid_answers, key=lambda x: x["score"], reverse=True)[0]
        else:
            # If no valid answer found, use the null prediction score
            best_answer = {"score": float('-inf'), "text": ""} # Use negative infinity for score if no valid answer

        # Compare best answer score with null score
        # We need the null score from the model's output, which is the score for the [CLS] token.
        # The null score is computed from the logits of the [CLS] token (usually index 0).
        # Let's re-calculate the null score based on the first token's logits across all features for this example.
        # A common practice is to take the max null score across all features for an example.
        null_scores = [all_start_logits[idx][0] + all_end_logits[idx][0] for idx in feature_indices]
        max_null_score = max(null_scores) if null_scores else float('-inf')


        # We need an ID for the prediction key. Since the original dataset doesn't have an 'id',
        # we can use the original example index as the ID for the prediction.
        # Compare best answer score with max null score
        if best_answer["score"] > max_null_score:
             predictions[str(example_index)] = best_answer["text"]
        elif max_null_score > (best_answer["score"] if valid_answers else float('-inf')):
             predictions[str(example_index)] = "" # Predict the null answer
        else:
             # If scores are equal, prefer the non-null answer
             predictions[str(example_index)] = best_answer["text"] if valid_answers else ""


    return predictions


# Define the QA metric calculation function
def compute_metrics(eval_predictions):
    # eval_predictions is a tuple: (predictions, label_ids)
    # predictions is a tuple: (start_logits, end_logits)
    # label_ids are the true start/end positions (not used for SQuAD v2 metric directly)

    start_logits, end_logits = eval_predictions.predictions

    # Access the original validation set and tokenized features
    # The Trainer passes the eval_dataset's features implicitly to compute_metrics.
    # We need to access the original validation_set which should be available in this scope
    # because this function is defined after validation_set is loaded.
    original_examples = validation_set # Use the original validation_set
    tokenized_features = tokenized_validation_set # Use the tokenized validation set

    # Post-process predictions using the original examples and tokenized features
    predictions = postprocess_qa_predictions(original_examples, tokenized_features, (start_logits, end_logits)) # Pass original examples and tokenized features

    # Reformat the ground truth answers for the metric function
    # Use the original examples and their indices as IDs
    references = [
        {"id": str(i), "answers": ex["answers"]} # Use string of index as ID for references
        for i, ex in enumerate(original_examples)
    ]

    # Load the metric (using the standard SQuAD v2 evaluation script)
    metric = evaluate.load("squad_v2")
    # metric_results = metric.compute(predictions=predictions, references=references)
    # print("Metric computation result:", metric_results) # Add print statement here
    # return metric_results # Return the actual results


    # --- Temporary: Return dummy metrics to test Trainer logging ---
    return {"exact_match": 0.5, "f1": 0.6}
    # --- End Temporary ---

In [ ]:
import collections

# Define training arguments (they are needed to initialize Trainer, but we are only using .evaluate())
args = TrainingArguments(
    output_dir="temp_validation",
    per_device_eval_batch_size=16, # Use a reasonable batch size
    report_to="wandb", # Explicitly report metrics to Weights & Biases
)

# Re-initialize the Trainer object
trainer = Trainer(
    model=model,
    args=args,
    tokenizer=tokenizer,
    eval_dataset=tokenized_validation_set,
    # Pass the original validation set to the trainer for use in compute_metrics
    # eval_examples=validation_set, # Removed invalid argument
    # Use the defined compute_metrics function
    compute_metrics=compute_metrics,
)

# Run the final evaluation
# This will output the F1, Exact Match, and other SQuAD V2 metrics
final_metrics = trainer.evaluate()

print("\n--- FINAL EVALUATION METRICS ---")
print(final_metrics)

In [ ]:
# Manual Evaluation to Debug Metrics

# Get raw predictions from the model
# This requires iterating through the tokenized validation set
# and getting the logits for each feature
eval_dataloader = trainer.get_eval_dataloader()
all_start_logits = []
all_end_logits = []

model.eval() # Set the model to evaluation mode
for batch in tqdm(eval_dataloader, desc="Generating predictions"):
    with torch.no_grad():
        # Move batch to the same device as the model
        batch = {k: v.to(model.device) for k, v in batch.items()}
        outputs = model(**batch)
        start_logits = outputs.start_logits
        end_logits = outputs.end_logits

        all_start_logits.append(start_logits.cpu().numpy())
        all_end_logits.append(end_logits.cpu().numpy())

all_start_logits = np.concatenate(all_start_logits)
all_end_logits = np.concatenate(all_end_logits)

raw_predictions = (all_start_logits, all_end_logits)

# Now, call the compute_metrics function manually
# We need to pass the original validation_set and tokenized_validation_set
# These should be available in the notebook's global scope

manual_metrics = compute_metrics(
    eval_predictions=type('obj', (object,), {'predictions': raw_predictions})() # Create a dummy object with predictions attribute
)

print("\n--- MANUAL EVALUATION METRICS ---")
print(manual_metrics)